# MobileMimic: Cross-Domain Generalization Demo

**This project demonstrates how to train and test a behavioral authentication system using the MobileMimic dataset**, the model trained on the legitimate user vs. random-stranger "zero-effort" impostors and evaluated against a harder adversary: someone who has studied and deliberately mimics that specific user's flicking behavior.

Please pay attention to the files `requirements.txt` and `README.md`, as well as the *raw MobileMimic CSV files from Figshare* (the download link and DOI can be found in the `README.md` file).


## What this notebook does

1. Loads raw per-event touch/orientation CSVs (18 fields per event) and extracts **per-flick features** (a "flick" is one continuous swipe gesture) : the 49-feature **Dynamic Data** representation.
2. Picks one victim, other victims, and one of that victim's real, video-informed mimicry attackers.
3. Trains a simple RBF-kernel SVM with victim's genuine swipes vs. a **balanced pool of other victims'** genuine flicks (a "zero-effort" impostor model).
4. Evaluates the model twice: 
   - against **held-out genuine zero-effort impostors** (people it was never trained on, but who are just being themselves)
   - against the victim's **real mimicry attacker** (someone who deliberately studied and copied this specific victim).


## Threat models: zero-effort impostor vs. targeted mimicry attacker

| | Who they are | How they behave |
|---|---|---|
| **Zero-effort impostor** | Any other person | Interacts naturally, using their *own* flicking behavior, not trying to imitate anyone |
| **Targeted mimicry attacker** | Someone who has studied a specific victim | Has watched recorded video of that victim's actual flicking behavior and deliberately tries to reproduce it |

The MobileMimic dataset pairs each victim's genuine baseline with real attackers who studied and imitated that specific victim, making the harder, more realistic comparison possible.


In [1]:
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

print("Environment OK.")


Environment OK.


## 1. Point this notebook at your downloaded copy of the dataset

Download and extract the MobileMimic dataset from Figshare (see `README.md` for the DOI
and citation). It is released as two subfolders of headerless CSVs:

```
<wherever you extracted it>/
├── victim/
│   ├── V01_victim.csv
│   ├── V02_victim.csv
│   └── ...
└── attack/
    ├── A01_attack_V01.csv
    ├── A01_attack_V02.csv
    └── ...
```

Set `DATA_ROOT` below to that folder.


In [2]:
DATA_ROOT = Path("./MobileMimic_Data")  # <-- EDIT THIS to point at your extracted download

VICTIM_DIR = DATA_ROOT / "victim"
ATTACK_DIR = DATA_ROOT / "attack"

if not VICTIM_DIR.exists() or not ATTACK_DIR.exists():
    raise FileNotFoundError(
        f"Couldn't find {VICTIM_DIR} and {ATTACK_DIR}. Download the dataset from Figshare "
        f"(see README.md), extract it, and point DATA_ROOT at the folder containing "
        f"victim/ and attack/ subfolders."
    )

VICTIM_RE = re.compile(r"^(V\d+)_victim\.csv$", re.IGNORECASE)
ATTACK_RE = re.compile(r"^(A\d+)_attack_(V\d+)\.csv$", re.IGNORECASE)

victim_ids = sorted(m.group(1).upper() for f in VICTIM_DIR.glob("*.csv")
                     if (m := VICTIM_RE.match(f.name)))
victim_to_sessions = {}
for f in sorted(ATTACK_DIR.glob("*.csv")):
    m = ATTACK_RE.match(f.name)
    if m:
        attacker_id, target_victim = m.group(1).upper(), m.group(2).upper()
        victim_to_sessions.setdefault(target_victim, []).append(f"{attacker_id}_attack_{target_victim}")

n_sessions = sum(len(v) for v in victim_to_sessions.values())
print(f"Found {len(victim_ids)} victims and {n_sessions} attacker sessions.")
print(f"Victims with at least one real attacker: {len(victim_to_sessions)}")


Found 23 victims and 113 attacker sessions.
Victims with at least one real attacker: 23


## 2. Per-flick feature extraction, from raw touch events

**The released dataset itself has 18 raw fields per event**. A **flick** is one continuous swipe gesture: a contiguous run of `down`/`move` rows sharing the same `(Session, TaskID, FlickCount)`. This section computes, per flick, the same kinematic and orientation feature set the dataset documentation calls the **Dynamic Data** representation: 16 touchscreen features (start/end position, path distance, elapsed time, tangential angle, curvature, velocity, acceleration, touch size and pressure) plus 33 orientation/accelerometer features (per-sample orientation and motion channels, and their per-flick average and standard deviation), **49 features in total per raw sample**.

If you only want to *use* the released dataset rather than understand exactly how these features are derived, you can skip ahead to Section 3 once you've run the two code cells below once.


In [3]:
# The released dataset's 18 raw fields per event
# everything computed below (the 49-feature "Dynamic Data" representation) is
# derived from these fields, not part of the raw release itself.
RAW_COLUMNS = [
    "ID", "Posture", "FlickType", "Session", "TaskID", "SpecificTaskNumber",
    "ClickNumber", "FlickCount", "FlickAction", "TimeMs", "TimeNs",
    "Pitch", "Roll", "Azimuth", "RawX", "RawY", "TouchPressure", "TouchSize",
]
assert len(RAW_COLUMNS) == 18

# The "Dynamic Data" 49-feature set used throughout the dataset paper's reference
# classifiers: each raw touch sample's own kinematics, plus its flick's per-flick
# mean (avg_*) and standard deviation (std_*) of the orientation/motion channels.
# This is a DERIVED representation computed from the 18 raw fields, it is
# not part of the released dataset itself.
SAMPLE_LEVEL_FEATURES = (
    ["XYID", "Dist", "HorizontalDist", "VerticalDist", "Tangential", "Curvature",
     "Velocity", "HorizontalVelocity", "VerticalVelocity",
     "Acceleration", "HorizontalAcceleration", "VerticalAcceleration",
     "AngularVelocity", "AngularAcceleration",
     "TouchSize", "TouchPressure", "Pitch", "Roll", "OriXY",
     "VelocityX", "VelocityY", "VelocityZ", "VelocityXY",
     "AccelerationX", "AccelerationY", "AccelerationZ", "AccelerationXY"]
)
DYNAMIC_DATA_FEATURES = (
    ["StartXID", "StartYID", "EndXID", "EndYID", "DistanceFlick",
     "HorizontalDistanceFlick", "VerticalDistanceFlick", "Path", "FlickArea"]
    + ["XYID", "Tangential", "Curvature", "Velocity", "Acceleration", "TouchSize", "TouchPressure"]
    + ["Pitch", "Roll", "OriXY", "VelocityX", "VelocityY", "VelocityZ", "VelocityXY",
       "AccelerationX", "AccelerationY", "AccelerationZ", "AccelerationXY"]
    + [f"avg_{c}" for c in ("Pitch", "Roll", "OriXY", "VelocityX", "VelocityY", "VelocityZ",
                             "VelocityXY", "AccelerationX", "AccelerationY", "AccelerationZ", "AccelerationXY")]
    + [f"std_{c}" for c in ("Pitch", "Roll", "OriXY", "VelocityX", "VelocityY", "VelocityZ",
                             "VelocityXY", "AccelerationX", "AccelerationY", "AccelerationZ", "AccelerationXY")]
)
assert len(DYNAMIC_DATA_FEATURES) == 49


def mat_round(x):
    '''Round-half-away-from-zero.'''
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.floor(np.abs(x) + 0.5)


def complete_values(values, time_rec):
    '''Fill exact-zero entries (velocity can be 0 due to sensor update-rate gaps) by
    linear interpolation between the nearest non-zero neighbors in time.'''
    values = np.array(values, dtype=float)
    nonzero_idx = np.flatnonzero(values != 0)
    if nonzero_idx.size == 0:
        return values
    for i in np.flatnonzero(values == 0):
        prev_c = nonzero_idx[nonzero_idx < i]
        post_c = nonzero_idx[nonzero_idx > i]
        prev_i = prev_c[-1] if prev_c.size else None
        post_i = post_c[0] if post_c.size else None
        if prev_i is not None and post_i is not None:
            t_a, t_b, t_c = time_rec[prev_i], time_rec[post_i], time_rec[i]
            values[i] = values[prev_i] + (values[post_i] - values[prev_i]) * (t_c - t_a) / (t_b - t_a)
        elif post_i is not None:
            values[i] = values[post_i]
        elif prev_i is not None:
            values[i] = values[prev_i]
    return values


def calculate_flick_area(coord_x, coord_y, sizes, grid_shape=(1080, 1920), radius_scale=235.0):
    '''Union pixel-area covered by circles (radius = touch size x radius_scale) centered
    at each touch point -- a rough proxy for how much screen area a flick's contact
    swept across.'''
    w, h = grid_shape
    canvas = np.zeros((w, h), dtype=bool)
    for px, py, sz in zip(np.asarray(coord_x, float), np.asarray(coord_y, float), np.asarray(sizes, float)):
        if np.isnan(px) or np.isnan(py) or np.isnan(sz):
            continue
        radius = sz * radius_scale
        if radius <= 0:
            continue
        x0, x1 = max(int(np.floor(px - radius)), 0), min(int(np.floor(px + radius)), w - 1)
        y0, y1 = max(int(np.floor(py - radius)), 0), min(int(np.floor(py + radius)), h - 1)
        if x0 > x1 or y0 > y1:
            continue
        xx, yy = np.meshgrid(np.arange(x0, x1 + 1), np.arange(y0, y1 + 1), indexing="ij")
        canvas[x0:x1 + 1, y0:y1 + 1] |= (xx - px) ** 2 + (yy - py) ** 2 <= radius ** 2
    return int(canvas.sum())


In [ ]:
def compute_flick_features(block: pd.DataFrame):
    '''Given one flick's contiguous rows, return a (n_samples, 49) array in DYNAMIC_DATA_FEATURES order,
    each raw sample's own kinematics plus this flick's per-flick aggregates,
    broadcast to every sample row.'''
    n = len(block)
    time_rec = block["TimeMs"].to_numpy(dtype=float).copy()

    if n >= 3:
        dt_raw = np.diff(time_rec)
        if np.std(dt_raw) > 0:
            lower = np.mean(dt_raw) - np.std(dt_raw)
            for i in np.flatnonzero(dt_raw < lower):
                if i == 0:
                    dt_raw[i] = (dt_raw[0] + dt_raw[1]) / 2
                elif i == len(dt_raw) - 1:
                    dt_raw[i] = (dt_raw[-2] + dt_raw[-1]) / 2
                else:
                    dt_raw[i] = (dt_raw[i - 1] + dt_raw[i] + dt_raw[i + 1]) / 3
            time_rec = np.concatenate([[time_rec[0]], time_rec[0] + np.cumsum(dt_raw)])

    pitch = -block["Pitch"].to_numpy(dtype=float)
    roll = -block["Roll"].to_numpy(dtype=float)
    azimuth = block["Azimuth"].to_numpy(dtype=float).copy()
    if np.any((azimuth > 90) & (azimuth <= 180)) and np.any((azimuth >= -180) & (azimuth < 90)):
        azimuth[(azimuth >= -180) & (azimuth < 0)] += 360
    raw_x = block["RawX"].to_numpy(dtype=float)
    raw_y = block["RawY"].to_numpy(dtype=float)
    touch_size = block["TouchSize"].to_numpy(dtype=float)
    touch_pressure = block["TouchPressure"].to_numpy(dtype=float)

    xy_id = mat_round(raw_x / 80.0) * 24 + mat_round(raw_y / 80.0)
    ori_xy = np.where(roll < 0, -np.hypot(pitch, roll), np.hypot(pitch, roll))

    dt = np.diff(time_rec)
    dx, dy = np.diff(raw_x), np.diff(raw_y)
    dist = np.hypot(dx, dy)
    with np.errstate(divide="ignore", invalid="ignore"):
        velocity, horiz_vel, vert_vel = dist / dt, dx / dt, dy / dt
        velocity_x, velocity_y = np.diff(pitch) / dt, np.diff(roll) / dt
        velocity_z, velocity_xy = np.diff(azimuth) / dt, np.diff(ori_xy) / dt
        tangential = np.degrees(np.arctan(dy / dx))
    tangential = tangential[~np.isnan(tangential)]
    velocity_x, velocity_y = complete_values(velocity_x, time_rec[:-1]), complete_values(velocity_y, time_rec[:-1])
    velocity_z, velocity_xy = complete_values(velocity_z, time_rec[:-1]), complete_values(velocity_xy, time_rec[:-1])

    if n >= 3:
        dt2 = np.diff(time_rec[:-1])
        with np.errstate(divide="ignore", invalid="ignore"):
            acceleration, horiz_accel, vert_accel = np.diff(velocity) / dt2, np.diff(horiz_vel) / dt2, np.diff(vert_vel) / dt2
            accel_x, accel_y = np.diff(velocity_x) / dt2, np.diff(velocity_y) / dt2
            accel_z, accel_xy = np.diff(velocity_z) / dt2, np.diff(velocity_xy) / dt2
    else:
        dt2 = np.array([])
        acceleration = horiz_accel = vert_accel = accel_x = accel_y = accel_z = accel_xy = np.array([])

    if tangential.size >= 2:
        seg_len = min(tangential.size - 1, dist.size)
        with np.errstate(divide="ignore", invalid="ignore"):
            curvature = np.diff(tangential)[:seg_len] / dist[:seg_len]
        curvature[np.isinf(curvature)] = np.nan
    else:
        curvature = np.array([])

    def pad(arr):
        arr = np.asarray(arr, dtype=float)
        return arr[:n] if len(arr) >= n else np.concatenate([arr, np.full(n - len(arr), np.nan)])

    # The 11 channels averaged/std'd over the whole flick (avg_*/std_* features).
    # Built as plain numpy arrays
  
    avg_std_source = {
        "Pitch": pitch, "Roll": roll, "OriXY": ori_xy,
        "VelocityX": pad(velocity_x), "VelocityY": pad(velocity_y),
        "VelocityZ": pad(velocity_z), "VelocityXY": pad(velocity_xy),
        "AccelerationX": pad(accel_x), "AccelerationY": pad(accel_y),
        "AccelerationZ": pad(accel_z), "AccelerationXY": pad(accel_xy),
    }
    with warnings.catch_warnings():
        # very short flicks (2-3 samples) can leave a channel all-NaN (e.g. no
        # acceleration is defined from only 2 points) 
        warnings.simplefilter("ignore", category=RuntimeWarning)
        avg = {k: np.nanmean(v) for k, v in avg_std_source.items()}
        std = {k: np.nanstd(v, ddof=1) for k, v in avg_std_source.items()}  # ddof=1 matches pandas .std()

    per_sample = {
        "XYID": xy_id, "Tangential": pad(tangential), "Curvature": pad(curvature),
        "Velocity": pad(velocity), "Acceleration": pad(acceleration),
        "TouchSize": touch_size, "TouchPressure": touch_pressure,
        **avg_std_source,  # Pitch/Roll/OriXY/VelocityX../AccelerationXY also appear per-sample, unaggregated
    }

    flick_area = calculate_flick_area(raw_x, raw_y, touch_size)
    h_dist, v_dist = raw_x[-1] - raw_x[0], raw_y[-1] - raw_y[0]
    distance_flick = float(np.hypot(h_dist, v_dist))
    flick_scalar = {
        "StartXID": mat_round(raw_x[0] / 80.0), "StartYID": mat_round(raw_y[0] / 80.0),
        "EndXID": mat_round(raw_x[-1] / 80.0), "EndYID": mat_round(raw_y[-1] / 80.0),
        "DistanceFlick": distance_flick, "HorizontalDistanceFlick": h_dist, "VerticalDistanceFlick": v_dist,
        "Path": float(np.nansum(dist)) if dist.size else 0.0, "FlickArea": flick_area,
    }

    columns = []
    for name in DYNAMIC_DATA_FEATURES:
        if name in flick_scalar:
            columns.append(np.full(n, flick_scalar[name]))
        elif name in per_sample:
            columns.append(per_sample[name])
        elif name.startswith("avg_"):
            columns.append(np.full(n, avg[name[4:]]))
        else:  # std_*
            columns.append(np.full(n, std[name[4:]]))

    return np.stack(columns, axis=1), distance_flick


### Turning per-flick feature tables into a fixed-size representation

Each flick has a different number of raw samples. To feed a classifier a fixed-length vector, every flick is truncated/padded to `TARGET_LEN = 10` samples (10 x 49 = 490 features when flattened), zero-padded if shorter, and truncated to the first 10 samples if longer. This is a simplified version of the full reference pipeline's approach (which also generates additional overlapping training windows from unusually long flicks); for demonstrating how to use the dataset, one window per flick keeps the code much easier to follow.


In [5]:
TARGET_LEN = 10


def extract_participant_flicks(csv_path: Path, target_len: int = TARGET_LEN):
    '''Raw CSV -> (X, sessions). X is (n_flicks, target_len, 49); sessions is the
    1-10 session index each flick belongs to, so you can split train/test by session.'''
    raw = pd.read_csv(csv_path, header=None)
    df = raw.iloc[:, :len(RAW_COLUMNS)].copy()
    df.columns = RAW_COLUMNS
    df = df.sort_values("TimeMs", kind="mergesort").reset_index(drop=True)
    df = df[(df["FlickType"] == "UpDown") & (df["FlickAction"].isin(["down", "move"]))]

    X_list, session_list = [], []
    for session, sdf in df.groupby("Session", sort=True):
        sdf = sdf.sort_values("TimeMs", kind="mergesort").reset_index(drop=True).copy()
        sdf["TimeMs"] = (sdf["TimeMs"] - sdf["TimeMs"].iloc[0]) / 1000.0
        changed = sdf["TaskID"].ne(sdf["TaskID"].shift()) | sdf["FlickCount"].ne(sdf["FlickCount"].shift())
        block_id = changed.cumsum()
        for _, block in sdf.groupby(block_id, sort=True):
            if len(block) < 2:
                continue
            feats, distance_flick = compute_flick_features(block)
            if not distance_flick or np.isnan(distance_flick):
                continue
            feats = np.nan_to_num(feats, nan=0.0)
            L = len(feats)
            window = feats[:target_len] if L >= target_len else np.vstack(
                [feats, np.zeros((target_len - L, feats.shape[1]))])
            X_list.append(window)
            session_list.append(int(session))

    X = np.stack(X_list) if X_list else np.empty((0, target_len, len(DYNAMIC_DATA_FEATURES)))
    return X, np.array(session_list, dtype=int)


# Quick test on one file
t0 = time.time()
_X_test, _sess_test = extract_participant_flicks(VICTIM_DIR / f"{victim_ids[0]}_victim.csv")
print(f"{victim_ids[0]}: {_X_test.shape[0]} flicks extracted in {time.time()-t0:.1f}s, shape={_X_test.shape}")


V01: 2043 flicks extracted in 6.2s, shape=(2043, 10, 49)


## 3. Set up the cross-domain generalization scenario

Pick a target victim, other victims for the zero-effort impostor for training.

In [6]:
N_OTHER_VICTIMS = 8   # how many other victims to include in the zero-effort training pool (adjustable)
TEST_SESSIONS = {9, 10}  # last 2 of each participant's 10 sessions held out as the test split

target_victim = next(v for v in victim_ids if v in victim_to_sessions)
target_attacker_session = victim_to_sessions[target_victim][0]
zero_effort_pool = [v for v in victim_ids if v != target_victim][:N_OTHER_VICTIMS]

print(f"Target victim:            {target_victim}")
print(f"Real mimicry attacker:    {target_attacker_session}")
print(f"Zero-effort train pool:   {zero_effort_pool}")

Target victim:            V01
Real mimicry attacker:    A01_attack_V01
Zero-effort train pool:   ['V02', 'V03', 'V04', 'V05', 'V06', 'V07', 'V08', 'V09']


In [7]:
def split_by_session(X, sessions, test_sessions):
    test_mask = np.isin(sessions, list(test_sessions))
    return X[~test_mask], X[test_mask]


def flatten(X):
    return X.reshape(X.shape[0], -1)


print("Extracting flicks for everyone involved in this scenario (one-time cost)...")
t0 = time.time()

X_victim, s_victim = extract_participant_flicks(VICTIM_DIR / f"{target_victim}_victim.csv")
pos_train, pos_test = split_by_session(X_victim, s_victim, TEST_SESSIONS)

X_attacker, s_attacker = extract_participant_flicks(ATTACK_DIR / f"{target_attacker_session}.csv")
_, neg_test_mimicry = split_by_session(X_attacker, s_attacker, TEST_SESSIONS)

zero_effort_train_flicks = {}
for v in zero_effort_pool:
    X_v, s_v = extract_participant_flicks(VICTIM_DIR / f"{v}_victim.csv")
    zero_effort_train_flicks[v], _ = split_by_session(X_v, s_v, TEST_SESSIONS)

print(f"Done in {time.time()-t0:.1f}s.")
print(f"  {target_victim} train/test flicks:        {len(pos_train)} / {len(pos_test)}")
print(f"  {target_attacker_session} test flicks:     {len(neg_test_mimicry)}")

Extracting flicks for everyone involved in this scenario (one-time cost)...


Done in 75.8s.
  V01 train/test flicks:        1696 / 347
  A01_attack_V01 test flicks:     546


### Balancing the zero-effort training pool

Concatenating every other victim's genuine flicks gives the impostor class far more rows than the victim's own.


In [8]:
def pool_balanced_by_participant(per_participant_flicks: dict, target_n: int, rng: np.random.Generator):
    sources = list(per_participant_flicks.values())
    if not sources or target_n <= 0:
        return np.empty((0,) + sources[0].shape[1:]) if sources else np.empty((0, TARGET_LEN, len(DYNAMIC_DATA_FEATURES)))
    quota = max(1, target_n // len(sources))
    sampled = []
    for arr in sources:
        n = len(arr)
        if n == 0:
            continue
        take = min(quota, n)
        idx = rng.choice(n, size=take, replace=False)
        sampled.append(arr[idx])
    return np.concatenate(sampled, axis=0) if sampled else np.empty((0,) + sources[0].shape[1:])


rng = np.random.default_rng(12345)
neg_train = pool_balanced_by_participant(zero_effort_train_flicks, target_n=len(pos_train), rng=rng)
print(f"pos_train={len(pos_train)}, neg_train (balanced)={len(neg_train)}")


pos_train=1696, neg_train (balanced)=1696


## 4. Train and evaluate

`compute_eer` finds the ROC operating point where the impostor-accepted rate (FAR) and genuine-user-rejected rate (FRR) are equal, the standard metric in the biometric authentication literature, since it doesn't require picking one decision threshold in advance. Lower EER is better; 50% is chance level.

In [ ]:
def compute_eer(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float((fpr[idx] + fnr[idx]) / 2)


X_train = np.concatenate([flatten(pos_train), flatten(neg_train)], axis=0)
y_train = np.concatenate([np.ones(len(pos_train)), np.zeros(len(neg_train))])

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

clf = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")
clf.fit(X_train_scaled, y_train)
print(f"Trained on {len(X_train)} flicks ({len(pos_train)} genuine, {len(neg_train)} zero-effort impostor).")


def evaluate(pos_test, neg_test, label):
    X_test = np.concatenate([flatten(pos_test), flatten(neg_test)], axis=0)
    y_test = np.concatenate([np.ones(len(pos_test)), np.zeros(len(neg_test))])
    scores = clf.decision_function(scaler.transform(X_test))
    eer = compute_eer(y_test, scores)
    print(f"{label}: EER = {eer*100:.2f}%  ({len(pos_test)} genuine vs. {len(neg_test)} impostor test flicks)")
    return eer


eer_mimicry = evaluate(pos_test, neg_test_mimicry, f"Mimicry test ({target_attacker_session}, real attacker)")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
bars = ax.bar(["Real mimicry\nattacker"], [eer_mimicry * 100], color=["#d03b3b"], width=0.5)
ax.axhline(50, color="#898781", linestyle=":", linewidth=1, label="Chance level (50%)")
for bar, val in zip(bars, [eer_mimicry]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{val*100:.1f}%",
            ha="center", fontsize=10)
ax.set_ylabel("Equal Error Rate (%)")
ax.set_title(f"{target_victim}: trained on zero-effort data, tested against a real mimicry attacker", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## 5. Citation

If you use this dataset, please cite the dataset descriptor paper and the Figshare dataset, see `README.md` in this folder for the full citation and DOI.
